In [10]:
# Cell 1: Imports and setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CanineModel, CanineTokenizer
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from classes.conversational_dataset import ConversationDataset
from classes.dual_encoder import DualEncoderModel
from tqdm import tqdm
import os
import random
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [11]:
# Cell 3: Dual-Contrastive Loss Function
# class DualContrastiveLoss(nn.Module):
#     """
#     Contrastive loss that operates on both content and style.
#     Forces model to separate content similarity from style similarity.
#     """
    
#     def __init__(self, temperature=0.1, alpha=0.7):
#         """
#         Args:
#             temperature: Softmax temperature
#             alpha: Weight for style loss (0.0 = only content, 1.0 = only style)
#         """
#         super().__init__()
#         self.temperature = temperature
#         self.alpha = alpha  # style loss weight
#         self.eps = 1e-8
    
#     def forward(self, content_emb, style_emb):
#         """
#         Args:
#             content_emb: (B, D_c) - Content embeddings
#             style_emb: (B, D_s) - Style embeddings
#         Returns:
#             Combined contrastive loss
#         """
#         batch_size = content_emb.size(0)
        
#         # Normalize embeddings
#         # content_emb = F.normalize(content_emb, dim=-1)
#         # style_emb = F.normalize(style_emb, dim=-1)
        
#         # --- CONTENT LOSS (InfoNCE) ---
#         # Positive pairs: same batch index (same text)
#         content_sim = torch.matmul(content_emb, content_emb.T) / self.temperature
#         content_labels = torch.arange(batch_size).to(content_emb.device)
#         content_loss = F.cross_entropy(content_sim, content_labels)
        
#         # --- STYLE LOSS (Style Discovery) ---
#         # Use SAME text but force style similarity to be meaningful
#         style_sim = torch.matmul(style_emb, style_emb.T) / self.temperature
        
#         # Create style targets based on content similarity
#         # (Texts with similar content should have similar style?)
#         # Or use uniform distribution for pure discovery
#         style_targets = F.softmax(style_sim.detach(), dim=-1)
        
#         # Symmetric style KL divergence
#         style_log_prob = F.log_softmax(style_sim, dim=-1)
#         style_loss = F.kl_div(style_log_prob, style_targets, reduction='batchmean')
        
#         # --- COMBINE LOSSES ---
#         total_loss = (1 - self.alpha) * content_loss + self.alpha * style_loss
        
#         return {
#             'total': total_loss,
#             'content': content_loss,
#             'style': style_loss
#         }

# Alternative simpler loss (choose one)
class StyleDiscoveryLoss(nn.Module):
    """
    Simplified version: Maximize similarity in style space.
    """
    
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, content_emb, style_emb):
        # Normalize
        # content_emb = F.normalize(content_emb, dim=-1)
        # style_emb = F.normalize(style_emb, dim=-1)
        
        # Compute similarity matrices
        content_sim = torch.matmul(content_emb, content_emb.T) / self.temperature
        style_sim = torch.matmul(style_emb, style_emb.T) / self.temperature
        
        # Remove diagonals (self-similarity)
        mask = torch.eye(content_emb.size(0), device=content_emb.device).bool()
        content_sim = content_sim.masked_fill(mask, -1e9)
        style_sim = style_sim.masked_fill(mask, -1e9)
        
        # Convert to probabilities
        content_prob = F.softmax(content_sim, dim=-1)
        style_prob = F.softmax(style_sim, dim=-1)
        
        # Cross-entropy between content and style distributions
        loss = -0.5 * (
            (content_prob * F.log_softmax(style_sim, dim=-1)).sum(dim=-1).mean() +
            (style_prob * F.log_softmax(content_sim, dim=-1)).sum(dim=-1).mean()
        )
        
        return loss

In [ ]:
# Cell 5: Load data and create dataloader
# Load data
rows = pd.read_csv("../data/train/retriever_train.csv")
rows['Content'] = rows['Content'].astype("string")
rows = rows.to_dict(orient="records")

# Initialize tokenizer and dataset
tokenizer = CanineTokenizer.from_pretrained("google/canine-s")
dataset = ConversationDataset(rows, tokenizer)

# Create dataloader
batch_size = 32
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")

Dataset size: 16000
Number of batches: 500


In [13]:
import math
from tqdm import tqdm

bad_rows = []

for idx, row in tqdm(enumerate(rows), total=len(rows)):
    content = row.get("Content", None)

    # Case 1: missing / null
    if content is None:
        bad_rows.append((idx, "None"))
        continue

    # Case 2: pandas <NA>
    if str(content) == "<NA>":
        bad_rows.append((idx, "pandas <NA>"))
        continue

    # Case 3: empty or whitespace
    if isinstance(content, str) and not content.strip():
        bad_rows.append((idx, "Empty string"))
        continue

    # Case 4: tokenizer-level failure
    try:
        tokenizer(
            content,
            truncation=True,
            max_length=128
        )
    except Exception as e:
        bad_rows.append((idx, f"Tokenizer error: {e}"))

print(f"\nFound {len(bad_rows)} problematic rows")

# Show first 10
for r in bad_rows[:10]:
    print(r)


100%|██████████| 16000/16000 [00:02<00:00, 7511.59it/s]


Found 0 problematic rows


In [ ]:
# Cell 6: Initialize model, loss, and optimizer
# Configuration
config = {
    "text_dim": 256,
    "style_dim": 64,
    "learning_rate": 3e-5,
    "content_lr": 1e-5,
    "style_lr": 5e-4,
    "temperature": 0.1,
    "alpha": 0.5,  # Style loss weight
    "epochs": 50,
    "save_dir": "../output/models/style_discovery_loss_full"
}

# Initialize model
model = DualEncoderModel(
    text_dim=config["text_dim"],
    style_dim=config["style_dim"]
).to(device)

# Initialize loss (choose one)
# criterion = DualContrastiveLoss(temperature=config["temperature"], alpha=config["alpha"])
criterion = StyleDiscoveryLoss(temperature=config["temperature"])

# Optimizer with different learning rates
optimizer = torch.optim.AdamW([
    {"params": model.base_encoder.parameters(), "lr": config["learning_rate"]},
    {"params": model.content_proj.parameters(), "lr": config["content_lr"]},
    {"params": list(model.style_proj.parameters()) + 
              list(model.style_mlp.parameters()), "lr": config["style_lr"]},
])

# Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=config["epochs"] * len(dataloader)
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model parameters: 132,945,280
Trainable parameters: 132,945,280


In [15]:
# Cell 7: Training loop
model.train()
os.makedirs(config["save_dir"], exist_ok=True)

for epoch in range(config["epochs"]):
    epoch_loss = 0.0
    epoch_content_loss = 0.0
    epoch_style_loss = 0.0
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{config['epochs']}")
    
    for batch_idx, batch in enumerate(pbar):
        # Move to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        content_emb, style_emb = model(input_ids, attention_mask)
        
        # Compute loss
        # if isinstance(criterion, DualContrastiveLoss):
        #     losses = criterion(content_emb, style_emb)
        #     loss = losses['total']
        #     content_loss = losses['content']
        #     style_loss = losses['style']
        # else:
        loss = criterion(content_emb, style_emb)
        content_loss = torch.tensor(0.0)
        style_loss = loss
        
        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        # Accumulate losses
        epoch_loss += loss.item()
        epoch_content_loss += content_loss.item()
        epoch_style_loss += style_loss.item()
        
        # Update progress bar
        pbar.set_postfix({
            "loss": loss.item(),
            "content": content_loss.item(),
            "style": style_loss.item()
        })
    
    # Epoch statistics
    avg_loss = epoch_loss / len(dataloader)
    avg_content_loss = epoch_content_loss / len(dataloader)
    avg_style_loss = epoch_style_loss / len(dataloader)
    
    print(f"[Epoch {epoch+1}] Total: {avg_loss:.4f}, "
          f"Content: {avg_content_loss:.4f}, Style: {avg_style_loss:.4f}")
    
    # Save checkpoint
    # if (epoch + 1) % 2 == 0 or epoch == config["epochs"] - 1:
    #     checkpoint_path = os.path.join(config["save_dir"], f"epoch_{epoch+1}")
    #     os.makedirs(checkpoint_path, exist_ok=True)
        
    #     # Save model state
    #     torch.save({
    #         'epoch': epoch,
    #         'model_state_dict': model.state_dict(),
    #         'optimizer_state_dict': optimizer.state_dict(),
    #         'loss': avg_loss,
    #         'config': config
    #     }, os.path.join(checkpoint_path, "model.pt"))
        
    #     print(f"Checkpoint saved to {checkpoint_path}")

print("Training complete!")

Epoch 1/5: 100%|██████████| 500/500 [02:30<00:00,  3.32it/s, loss=1.64, content=0, style=1.64]


[Epoch 1] Total: 1.7749, Content: 0.0000, Style: 1.7749


Epoch 2/5: 100%|██████████| 500/500 [02:29<00:00,  3.34it/s, loss=1.2, content=0, style=1.2]  


[Epoch 2] Total: 1.5040, Content: 0.0000, Style: 1.5040


Epoch 3/5: 100%|██████████| 500/500 [02:29<00:00,  3.33it/s, loss=1.06, content=0, style=1.06]  


[Epoch 3] Total: 1.2661, Content: 0.0000, Style: 1.2661


Epoch 4/5: 100%|██████████| 500/500 [02:29<00:00,  3.33it/s, loss=1.15, content=0, style=1.15]  


[Epoch 4] Total: 1.1501, Content: 0.0000, Style: 1.1501


Epoch 5/5: 100%|██████████| 500/500 [02:30<00:00,  3.33it/s, loss=1.16, content=0, style=1.16]  

[Epoch 5] Total: 1.0856, Content: 0.0000, Style: 1.0856
Training complete!


In [16]:
# Cell 8: Save final model
# Save final model
# final_path = os.path.join(config["save_dir"], "final")
final_path = config["save_dir"]
os.makedirs(final_path, exist_ok=True)

# Save model weights
torch.save(model.state_dict(), os.path.join(final_path, "dual_encoder.pt"))

# Save HuggingFace compatible model
model.base_encoder.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)

# Save projection heads
torch.save({
    'content_proj': model.content_proj.state_dict(),
    'style_proj': model.style_proj.state_dict(),
    'style_mlp': model.style_mlp.state_dict()
}, os.path.join(final_path, "projection_head.pt"))

print(f"Model saved to {final_path}")

Model saved to ../output/models/style_discovery_loss_full
